# CSCI 6353 · Topic 4 — Planning in a Known MDP

Iterative Policy Evaluation, Policy Iteration, and Value Iteration on the 4×4 GridWorld.

- **16 states** ($s_0$ start, $s_{15}$ terminal), 4 actions (up/down/left/right), reward **−1 per step**, walls keep you in place, $\gamma = 1$.
- Because the MDP is **known**, everything below is pure computation — no interaction, no episodes.

*Dr. Dongchul Kim · Summer II 2026*


## The model

We encode the dynamics as a simple function: `next_state(s, a)` returns where action `a` leads from state `s`. This *is* the transition model $P(s'\mid s,a)$ (deterministic here), and the reward model is simply $R = -1$ for every move.


In [ ]:
N = 16
TERMINAL = 15
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right

def next_state(s, a):
    """Deterministic transition model of the 4x4 GridWorld."""
    if s == TERMINAL:
        return s
    r, c = divmod(s, 4)
    dr, dc = ACTIONS[a]
    nr, nc = r + dr, c + dc
    if nr < 0 or nr > 3 or nc < 0 or nc > 3:   # wall: stay in place
        return s
    return nr * 4 + nc

def show(V, nd=2):
    """Print a value table as a 4x4 grid."""
    for r in range(4):
        print(" ".join(f"{V[r*4+c]:7.{nd}f}" for c in range(4)))


## 1. Iterative Policy Evaluation (IPE)

Evaluate the **uniform random policy** ($\pi(a\mid s) = 1/4$) by sweeping the Bellman expectation update until the table converges:

$$V_{k+1}(s) \leftarrow \sum_a \pi(a \mid s)\big( -1 + V_k(s') \big)$$


In [ ]:
def ipe(sweeps=None, tol=1e-10):
    """Evaluate the random policy. Stop after `sweeps` sweeps, or on convergence."""
    V = [0.0] * N
    k = 0
    while True:
        newV = [0.0] * N
        for s in range(N):
            if s == TERMINAL:
                continue                       # terminal keeps value 0
            newV[s] = sum(0.25 * (-1 + V[next_state(s, a)]) for a in range(4))
        k += 1
        diff = max(abs(newV[s] - V[s]) for s in range(N))
        V = newV
        if sweeps and k >= sweeps:
            return V, k
        if not sweeps and diff < tol:
            return V, k

for k in (1, 2, 3):
    Vk, _ = ipe(sweeps=k)
    print(f"k = {k}:")
    show(Vk)
    print()


k = 1:
  -1.00   -1.00   -1.00   -1.00
  -1.00   -1.00   -1.00   -1.00
  -1.00   -1.00   -1.00   -1.00
  -1.00   -1.00   -1.00    0.00

k = 2:
  -2.00   -2.00   -2.00   -2.00
  -2.00   -2.00   -2.00   -2.00
  -2.00   -2.00   -2.00   -1.75
  -2.00   -2.00   -1.75    0.00

k = 3:
  -3.00   -3.00   -3.00   -3.00
  -3.00   -3.00   -3.00   -2.94
  -3.00   -3.00   -2.88   -2.44
  -3.00   -2.94   -2.44    0.00



Exactly the tables from the lecture: the goal's influence spreads **one cell per sweep**. Now run to convergence:


In [ ]:
V_pi, iters = ipe()
print(f"converged after {iters} sweeps:")
show(V_pi, 1)


converged after 1161 sweeps:
  -59.4   -57.4   -54.3   -51.7
  -57.4   -54.6   -49.7   -45.1
  -54.3   -49.7   -40.9   -30.0
  -51.7   -45.1   -30.0     0.0



A randomly wandering agent needs on average **~59 steps** from the start. Note it took **1161 sweeps** to fully converge — motivation for early stopping.


## 2. Policy Iteration (with early stopping)

Alternate **evaluation** (a few IPE sweeps for the *current* policy) and **greedy improvement**. Even truncated evaluation (`eval_sweeps=5`) reaches the optimal policy in a couple of rounds.


In [ ]:
def evaluate(policy, sweeps=5):
    """A few Bellman-expectation sweeps for the given policy (early stopping)."""
    V = [0.0] * N
    for _ in range(sweeps):
        newV = [0.0] * N
        for s in range(N):
            if s == TERMINAL:
                continue
            newV[s] = sum(p * (-1 + V[next_state(s, a)])
                          for a, p in enumerate(policy[s]))
        V = newV
    return V

def greedy(V):
    """Greedy (deterministic) policy w.r.t. a value table."""
    policy = []
    for s in range(N):
        q = [-1 + V[next_state(s, a)] for a in range(4)]
        best = q.index(max(q))
        policy.append([1.0 if a == best else 0.0 for a in range(4)])
    return policy

# start from the uniform random policy
policy = [[0.25] * 4 for _ in range(N)]
for round in range(1, 6):
    V = evaluate(policy, sweeps=5)
    new_policy = greedy(V)
    changed = sum(p != q for p, q in zip(policy, new_policy))
    print(f"round {round}: policy changed in {changed} states")
    if changed == 0:
        break
    policy = new_policy


round 1: policy changed in 16 states
round 2: policy changed in 5 states
round 3: policy changed in 0 states



## 3. Value Iteration

Merge the two stages: replace the policy average with a **max** over actions (Bellman optimality update):

$$V(s) \leftarrow \max_a \big( -1 + V(s') \big)$$


In [ ]:
def value_iteration(tol=1e-10):
    V = [0.0] * N
    k = 0
    while True:
        newV = [0.0] * N
        for s in range(N):
            if s == TERMINAL:
                continue
            newV[s] = max(-1 + V[next_state(s, a)] for a in range(4))
        k += 1
        diff = max(abs(newV[s] - V[s]) for s in range(N))
        V = newV
        if diff < tol:
            return V, k

V_star, k = value_iteration()
print(f"value iteration converged after {k} sweeps:")
show(V_star, 0)


value iteration converged after 7 sweeps:
     -6      -5      -4      -3
     -5      -4      -3      -2
     -4      -3      -2      -1
     -3      -2      -1       0



**7 sweeps** instead of 1161 — and the values are the negative **shortest-path distances** to the goal ($V^*(s_0) = -6$). Compare with the random policy's $-59.4$. Finally, read off the optimal policy:


In [ ]:
ARROWS = ["↑", "↓", "←", "→"]
print("greedy policy w.r.t. V*:")
for r in range(4):
    row = []
    for c in range(4):
        s = r * 4 + c
        if s == TERMINAL:
            row.append("G")
            continue
        q = [-1 + V_star[next_state(s, a)] for a in range(4)]
        row.append(ARROWS[q.index(max(q))])
    print(" ".join(row))


greedy policy w.r.t. V*:
↓ ↓ ↓ ↓
↓ ↓ ↓ ↓
↓ ↓ ↓ ↓
→ → → G



Every state walks toward the goal. (Many states have **ties** — e.g. moving right is equally optimal in the interior; `q.index(max(q))` just picks the first maximizer.)

---
**Try it yourself:**
1. Change $\gamma$ to 0.9 in IPE — how do the values change?
2. Put the terminal state at $s_0$ *and* $s_{15}$ — what does $V^*$ look like?
3. Add a "wind" that pushes the agent right with probability 0.2 — which cells change their optimal action?
